# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [55]:
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30,
               STDDEV_SAMP(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_volatility_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 10
    )
    SELECT * FROM windowed
""").df()

# STDDEV_SAMP returns NULL for content items with only 1 non-null position reading in the window
# (can't compute variance from a single point) -- treat that as 'no observed volatility', i.e. 0,
# rather than dropping the row later in dropna().
features_90d['pos_volatility_last30'] = features_90d['pos_volatility_last30'].fillna(0)

print(f'{len(features_90d):,} content items with enough history (90-day window)')
assert len(features_90d) > 0, (
    'features_90d came back empty -- lower HAVING further, widen the window, or switch '
    "TABLES['fact_daily'] back to TABLES['fact_daily'] for the final pass."
)
features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

192,098 content items with enough history (90-day window)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,pos_volatility_last30
0,client_62f4a7e64f5e0096,content_a1a50eaf1c9738fb,123.0,584.0,0.0,60.810969,29.549958
1,client_62f4a7e64f5e0096,content_8f63ac570bf4dacf,232.0,331.0,0.0,45.253473,34.608581
2,client_62f4a7e64f5e0096,content_b8172b7cc8361d26,859.0,1331.0,6.0,10.532614,6.963721
3,client_62f4a7e64f5e0096,content_e932e5d70bd9bcd1,1036.0,2602.0,8.0,6.243032,3.396181
4,client_62f4a7e64f5e0096,content_67ad8e2137baf889,456.0,4273.0,0.0,15.632060,10.883740


### 6.2 — Rejoin query signals and rebuild the label

Same merge as section 4, just against the new 90-day feature table.

In [52]:
data_90d = features_90d.merge(qsignals, on='content_hash_id', how='left')
data_90d['is_declining'] = (data_90d['imp_last30'] < 0.8 * data_90d['imp_prev30']).astype(int)

feature_cols_v2 = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share',
                   'top_query_share', 'pos_volatility_last30']
model_data_90d = data_90d.dropna(subset=feature_cols_v2)

print(f'joined: {len(data_90d):,} rows')
print(f'modelable after dropna: {len(model_data_90d):,} rows')
assert len(model_data_90d) > 0, (
    'model_data_90d is empty after dropna -- check data_90d[feature_cols_v2].isna().sum() '
    'to see whether the HAVING filter or the qsignals merge is the cause.'
)
model_data_90d.head()

joined: 192,098 rows
modelable after dropna: 128,519 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,pos_volatility_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,is_declining
0,client_62f4a7e64f5e0096,content_7fbcb5910fea2c2c,355.0,1011.0,0.0,9.245561,6.443509,6.0,0.131040,0.784041,46.0,116.0,0.396552,1
1,client_62f4a7e64f5e0096,content_238db9c1360fe5d9,20.0,517.0,0.0,68.884615,25.158331,7.0,0.135940,0.564246,58.0,161.0,0.360248,1
3,client_62f4a7e64f5e0096,content_da06305afa2f68e2,187.0,727.0,0.0,22.437315,18.071202,6.0,0.292123,0.563457,60.0,132.0,0.454545,1
5,client_62f4a7e64f5e0096,content_080688632ba87fa7,1595.0,6407.0,8.0,5.860446,1.114220,6.0,0.008623,0.963884,92.0,220.0,0.418182,1
8,client_62f4a7e64f5e0096,content_515c48120ecf8a9d,218.0,586.0,0.0,7.847710,12.985937,2.0,0.136816,0.825871,16.0,30.0,0.533333,1


### 6.3 — Random split (baseline) vs. GroupShuffleSplit on `client_hash_id`

A random split can let the model see content items from the *same client* in both train and test. Since content from one client shares site-wide patterns (template, niche, seasonality), the model can partly memorize "how this client's pages behave" rather than learning something that generalizes to a client it has never seen. `GroupShuffleSplit` with `groups=client_hash_id` keeps every row from a given client entirely in train *or* entirely in test, which is a fairer test of whether the signal generalizes. If the group-split score drops noticeably versus the random split, that gap **is** the client-leakage the random split was hiding.

In [53]:
from sklearn.model_selection import GroupShuffleSplit

X = model_data_90d[feature_cols_v2]
y = model_data_90d['is_declining']
groups = model_data_90d['client_hash_id']

print(f'clients represented: {groups.nunique()}')

# --- Baseline: random split (same shape as section 5) ---
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print('=== Random split (content items can straddle clients) ===')
print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model_random.predict(X_te), digits=3))

# --- GroupShuffleSplit: entire clients held out ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert train_clients.isdisjoint(test_clients), 'client leaked across the group split'

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)

print('\n=== GroupShuffleSplit on client_hash_id (clients never seen in both) ===')
print(f'train clients: {len(train_clients)}   test clients: {len(test_clients)}')
print(f'base rate (always predict majority): {max(y_te_g.mean(), 1 - y_te_g.mean()):.3f}')
print(classification_report(y_te_g, model_grouped.predict(X_te_g), digits=3))


clients represented: 52
=== Random split (content items can straddle clients) ===
base rate (always predict majority): 0.857
              precision    recall  f1-score   support

           0      0.835     0.489     0.617      4590
           1      0.920     0.984     0.951     27540

    accuracy                          0.913     32130
   macro avg      0.878     0.736     0.784     32130
weighted avg      0.908     0.913     0.903     32130


=== GroupShuffleSplit on client_hash_id (clients never seen in both) ===
train clients: 39   test clients: 13
base rate (always predict majority): 0.838
              precision    recall  f1-score   support

           0      0.912     0.586     0.713      2573
           1      0.925     0.989     0.956     13347

    accuracy                          0.924     15920
   macro avg      0.919     0.787     0.835     15920
weighted avg      0.923     0.924     0.917     15920



### 6.4 — Which feature carries the signal?

Feature importances from the group-split model — the one that's harder to fool with client-level memorization, so its importances are more trustworthy than the random-split model's.

In [54]:
import pandas as pd

importances = pd.Series(
    model_grouped.feature_importances_, index=feature_cols_v2
).sort_values(ascending=False)

print('Feature importances (group-split model):')
print(importances.to_string(float_format=lambda x: f'{x:.3f}'))

Feature importances (group-split model):
imp_prev30              0.261
pos_volatility_last30   0.206
anon_share              0.167
rare_share              0.144
visible_queries         0.112
top_query_share         0.111
